# 6. Preprocesamiento y tokenización

<a target="_blank" href="https://colab.research.google.com/github/umoqnier/cl-2026-2-lab/blob/main/notebooks/6_tokenization.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<img src="https://2.bp.blogspot.com/-oDvCIkIjwXw/VdWWxfvmq3I/AAAAAAAARUE/r0MrmbNzMz8/s1600/inputoutput.jpg" width=500>

## Objetivos

- Listar algunos pasos comúnes para el preprocesamiento de texto
  - Aplicar preprocesamiento a un corpus
- Entender el funcionamiento de algoritmos de sub-word tokenization
  - BPE
  - Word-piece
  - Sentecepiece
- Entrenar modelos para sub-word tokenization
  - Aplicar BPE a corpus

## El lenguaje, datos inherentemente desarreglados

Los datos con los que trabajamos son inherentemente **desestructurados** y en general contienen ruido, irregularidades, e inconsistencias.

Los modelos de NLP son sensibles a estos problemas y en realidad no pueden utilizar el texto directamente. Dado que el texto es una representación simbólica de la información, es necesario convertirlo a una representación numérica que el modelo pueda utilizar. Esta representación generalmente serán **vectores** con valores reales llamados *embeddings*.

El objetivo es crear un *pipeline* que transforme el texto crudo en *embeddigs* que serán utilizados como entrada para crear modelos que resuelvan alguna tarea de NLP.

### *Pipelines*

![](https://i.makeagif.com/media/11-05-2015/x60GaR.gif)

Al crear sistemas de *NLP* nos enfrentamos con problemas complejos. Conviene entonces separar dichos problemas en pequeños problemas que podamos manejar y resolver por separado. 

El preprocessamiento es el primer paso de este proceso. Otros elementos pueden ser los siguientes:

- Definición del problema
- Adquisición de datos
- Ingeniería de características (*feature engineering*)
- Modelado
    - Definición de hiperparametros
- Entrenamiento
- Evaluación
- Puesta en producción
- Monitorización y actualización del modelo

## Elementos del preprocesamiento

- Limpieza del texto
    - Quitar etiquetas de marcado (HTML, XML, MD), metadatos y asegurarnos que todo esta en UTF-8
    - Eliminar header, footers o titulos que no aportan información
- Normalización
    - Pasar todo a minúsculas
    - Pasar texto a cierta norma ortográfica
    - Expansión de contracciones o abreviaciones
- Quitar stopwords y lematización/stemming
- Tokenización
    - Por palabra
    - Por letras
    - Por sub-palabras
- *Embeddings*
    - Los modelos solo entienden números, por lo que hay que convertir el texto a una representación vectorial

### Limpieza de textos

Es común usar regex o bibliotecas como `BeautifulSoup` para limpiar el texto de etiquetas de marcado

In [1]:
import requests
from bs4 import BeautifulSoup
from rich import print as rprint

In [2]:
url = "https://elotl.mx/blog/index.html"
response = requests.get(url)

soup = BeautifulSoup(response.content, "html.parser")
posts = soup.find("div", class_="widget_onetone_recent_posts")

if posts:
    rows = posts.find_all("li")
    for row in rows:
        text = row.get_text()
        rprint(text)

Presentación de Kolo    
    February 25, 2021

Esquite    
    July 29, 2020

Proyecto Tsu̱nkua    
    March 1, 2019

Comunidad Elotl: Iniciativa tecnológica que difunde lenguas indígenas (mexico.com)    
    February 20, 2019

Procesando el otomí (hñähñu) ¿Dónde empezar?    
    January 8, 2019

Búsqueda automática de traducciones para lenguas originarias de México    
    October 10, 2018

In [3]:
import nltk

nltk.download("gutenberg")

[nltk_data] Downloading package gutenberg to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


True

In [4]:
from nltk.corpus import gutenberg

moby = gutenberg.raw("melville-moby_dick.txt")

In [5]:
print(moby[:30000])

[Moby Dick by Herman Melville 1851]


ETYMOLOGY.

(Supplied by a Late Consumptive Usher to a Grammar School)

The pale Usher--threadbare in coat, heart, body, and brain; I see him
now.  He was ever dusting his old lexicons and grammars, with a queer
handkerchief, mockingly embellished with all the gay flags of all the
known nations of the world.  He loved to dust his old grammars; it
somehow mildly reminded him of his mortality.

"While you take in hand to school others, and to teach them by what
name a whale-fish is to be called in our tongue leaving out, through
ignorance, the letter H, which almost alone maketh the signification
of the word, you deliver that which is not true." --HACKLUYT

"WHALE. ... Sw. and Dan. HVAL.  This animal is named from roundness
or rolling; for in Dan. HVALT is arched or vaulted." --WEBSTER'S
DICTIONARY

"WHALE. ... It is more immediately from the Dut. and Ger. WALLEN;
A.S. WALW-IAN, to roll, to wallow." --RICHARDSON'S DICTIONARY

KETOS,               GRE

In [5]:
import re
from nltk.tokenize import sent_tokenize


def clean_and_extract_sentences(text: str) -> list[str]:
    """Clean preamble text from Moby Dick and extract sentences from the novel."""
    novel_start_patterns = [
        r"CHAPTER\s+1\b",  # Matches "CHAPTER 1"
        r"Call\s+me\s+Ishmael",  # Matches "Call me Ishmael"
    ]

    # Buscamos el índice donde comienza la novelaS
    novel_start = None
    for pattern in novel_start_patterns:
        match = re.search(pattern, text)
        if match and (novel_start is None or match.start() < novel_start):
            novel_start = match.start()

    if novel_start is None:
        return []

    # Descartamos el preambulo
    novel_text = text[novel_start:]

    # Limpiamos el texto
    novel_text = re.sub(r"CHAPTER\s+\d+", "", novel_text)
    novel_text = re.sub(r"\s+", " ", novel_text).strip()
    novel_text = re.sub(r"\[.*?\]", "", novel_text)

    # Extraemos las oraciones
    sentences = sent_tokenize(novel_text)

    # Limpieza adicional
    cleaned_sentences = []
    for sentence in sentences:
        sentence = sentence.strip()
        sentence = re.sub(r"^[^a-zA-Z]+", "", sentence)
        sentence = re.sub(r"[^a-zA-Z]+$", "", sentence)

        if sentence:
            cleaned_sentences.append(sentence)

    return cleaned_sentences

In [7]:
sentences = clean_and_extract_sentences(moby)
for i, sentence in enumerate(sentences[:10], 1):
    rprint(f"{i}. {sentence}")

1. Loomings

2. Call me Ishmael

3. Some years ago--never mind how long precisely--having little or no money in my purse, and nothing particular to 
interest me on shore, I thought I would sail about a little and see the watery part of the world

4. It is a way I have of driving off the spleen and regulating the circulation

5. Whenever I find myself growing grim about the mouth; whenever it is a damp, drizzly November in my soul; 
whenever I find myself involuntarily pausing before coffin warehouses, and bringing up the rear of every funeral I 
meet; and especially whenever my hypos get such an upper hand of me, that it requires a strong moral principle to 
prevent me from deliberately stepping into the street, and methodically knocking people's hats off--then, I account
it high time to get to sea as soon as I can

6. This is my substitute for pistol and ball

7. With a philosophical flourish Cato throws himself upon his sword; I quietly take to the ship

8. There is nothing surprising in this

9. If they but knew it, almost all men in their degree, some time or other, cherish very nearly the same feelings 
towards the ocean with me

10. There now is your insular city of the Manhattoes, belted round by wharves as Indian isles by coral 
reefs--commerce surrounds it with her surf

### Normalización

<center><img src="https://external-content.duckduckgo.com/iu/?u=http%3A%2F%2Fimg1.wikia.nocookie.net%2F__cb20140504152558%2Fspongebob%2Fimages%2Fe%2Fe3%2FThe_spongebob.jpg&f=1&nofb=1&ipt=28368023b54a7c84c9100025981b1042d0f4ca3ceaac53be42094cc1c3794348&ipo=images" height=300 width=300></center>

In [6]:
import unicodedata


def strip_accents(s: str) -> str:
    """Remove diacritical marks from characters in a Unicode string.

    Uses Unicode NFD (Normalization Form Decomposition) normalization to decompose accented characters into their
    base character + combining mark, then filters out combining marks (Mark, Nonspacing, Mn category).
    """
    return "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )

In [7]:
rprint(
    strip_accents("""Éxtasis
E-, e-, e-, e-, e-, e-, e-
Éxtasis
Éxtasis

Aquí no existe el bajón
Tamos' de fiestón, ya sabes
Despierta la inspiración
Si sacamos las suaves""")
)

Extasis
E-, e-, e-, e-, e-, e-, e-
Extasis
Extasis

Aqui no existe el bajon
Tamos' de fieston, ya sabes
Despierta la inspiracion
Si sacamos las suaves

- https://www.unicode.org/reports/tr44/#GC_Values_Table

> And keep in mind, these manipulations may significantly alter the meaning of the text. Accents, Umlauts etc. are not "decoration".
- [oefe](https://stackoverflow.com/users/49793/oefe) - [source](https://stackoverflow.com/questions/517923/what-is-the-best-way-to-remove-accents-normalize-in-a-python-unicode-string)

#### ¿Para otras lenguas?

- No hay muchos recursos :(
- Pero para el nahuatl esta `pyelotl` :)

#### Normalizando el Nahuatl

In [ ]:
!pip install elotl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 1.8 MB/s eta 0:00:00


In [18]:
import elotl.corpus
import elotl.nahuatl.orthography

In [19]:
axolotl = elotl.corpus.load("axolotl")

In [20]:
# Tres posibles normalizadores: sep, inali, ack
# Sauce: https://pypi.org/project/elotl/

nahuatl_normalizer = elotl.nahuatl.orthography.Normalizer("sep")

In [21]:
rprint(axolotl[1][1])

¿In chalchihuitl, teocuitlatl, mach ah ca on yaz?

In [22]:
rprint(nahuatl_normalizer.normalize(axolotl[1][1]))

¿in chalchiuitl, teokuitlatl, mach aj ka on yas?

In [24]:
nahuatl_normalizer.to_phones(axolotl[1][1])

'¿in t͡ʃalt͡ʃiwiƛ, teokʷiƛaƛ, mat͡ʃ aʔ ka on yas?'

### Stopwords

In [44]:
from nltk.corpus import stopwords

In [45]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [46]:
rprint(stopwords.words("spanish")[:15])

['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para', 'con']

### Definiendo una función de preprocesamiento

In [61]:
def preprocess(words: list[str], regex: str=r"[^\w+]", lang: str="en", remove_stops: bool = False, remove_accents: bool = False) -> list[str]:
    """Preprocess step for list of words in corpus
    """
    stop_lang = "english" if lang=="en" else "spanish"
    result = []
    for word in words:
        word = re.sub(regex, "", word).lower()
        if remove_stops and word in stopwords.words(stop_lang) or (len(word) < 2):
            continue
        
        if remove_accents:
            word = strip_accents(word)
        result.append(word)
    return result

## ¿Cuántas palabras hay en las siguientes oraciones?

In [ ]:
sentence_trapo = "Quitan el trapo y no lo ponen. ¿Por qué quitan el trapo? Si es una cosa que debe estar ahí."

In [ ]:
sentence_sad = "Mmmmm haz lo que quieras... pero no me digas que no te lo advertí 😓"

- A estas alturas tenemos cierta información acerca de las palabras:
    - **typos:** Número de palabras únicas en un corpus. *AKA* vocabulario
    - **tokens:** Número total de palabras. *AKA* instancias

## ¿Qué es una palabra?

- Técnicas de procesamiento del lenguaje depende de las palabras y las oraciones.
  - Debemos identificar estos elementos para poder procesarlos
- Este paso de identificación de palabras y oraciones se le llama segmentación de texto o **tokenización** (*tokenization*)
- Además de la identificación de unidades aplicaremos transformaciones al texto

### Más que mil palabras

Aunque la definición de lo que es una palabra puede parecer obvia a la hora de diseñar sistemas de PLN puede ser tremendamente difícil.

- I'm
- we'd
- I've
- Diego's Bicycle

En lenguas donde los espacios no son utilizados para marcar posibles delimitaciones entre palabras la cosa se pone más dura:

- 姚明进入总决赛 - yáo míng jìn rù zong jué sài
- *"Yao Ming llegó a las finales"*

> Tomado de (Jurafsky, 2026)

Chinese Treebank:

1. 姚明 - Yao Ming
2. 进入 - llego a
3. 总决赛 - finales

Peking University:

1. 姚 - Yao
2. 明 - Ming
3. 进入 - llego
4. 总 - generales
5. 决赛 - finales

Caracteres como límites

1. 姚 - Yao
2. 明 - Ming
3. 进 - entrar
4. 入 - entrar
5. 总 - generales
6. 决 - decisión
7. 赛 - juego

Otro problema a considerar es la cantidad de palabras con la que tendran que lidiar los modelos que diseñemos. Por más texto que tengamos a disposición siempre habrán palabras que el modelo no habrá visto (*AKA* **Out of Vocabulary, OOV** o **\<UNK\>**)

### Recordando los morfemas

- Con la morfología podemos identificar como se modifica el significado variando la estructura de las palabras
- Tambien las reglas para producir:
    - niño -> niños
    - niño -> niña
- Tenemos elementos mínimos, intercambiables que varian el significado de las palabras: **morfemas**

> Un morfema es la unidad mínima con significado en la producción lingüística (Mijangos, 2020)

#### Tipos de morfemas

- Bases: Subcadenas que aportan información léxica de la palabra
    - sol
    - frasada
- Afijos: Subcadenas que se adhieren a las bases para añadir información (flexiva, derivativa)
    - Prefijos
        - *in*-parable
    - Subfijos
        - pan-*ecitos*, come-*mos*

## Tokenización

### Word-base tokenization

In [1]:
text = """
¡¡¡Mamá prendele a la grabadora!!!, ¿llamaste a las vecinas? Corre la voz porque, efectivamente, !estoy en la tele! 📺
"""

In [2]:
text.split()

['¡¡¡Mamá',
 'prendele',
 'a',
 'la',
 'grabadora!!!,',
 '¿llamaste',
 'a',
 'las',
 'vecinas?',
 'Corre',
 'la',
 'voz',
 'porque,',
 'efectivamente,',
 '!estoy',
 'en',
 'la',
 'tele!',
 '📺']

In [ ]:
# [a-zA-Z_]
regex = r"\w+"
re.findall(regex, text)

['Mamá',
 'prendele',
 'a',
 'la',
 'grabadora',
 'llamaste',
 'a',
 'las',
 'vecinas',
 'Corre',
 'la',
 'voz',
 'porque',
 'efectivamente',
 'estoy',
 'en',
 'la',
 'tele']

In [19]:
re.findall(regex, "El valor de PI es 3.14159")

['El', 'valor', 'de', 'PI', 'es', '3', '14159']

<img src="http://images.wikia.com/battlebears/images/2/2c/Troll_Problem.jpg" with="250" height="250">

- Vocabularios gigantescos difíciles de procesar
- Generalmente, entre más grande es el vocabulario más pesado será nuestro modelo

**Ejemplo:**
- Si queremos representaciones vectoriales de nuestras palabras obtendríamos vectores distintos para palabras similares
    - niño = `v1(39, 34, 5,...)`
    - niños = `v2(9, 4, 0,...)`
    - niña = `v3(2, 1, 1,...)`
    - ...
- Tendríamos tokens con bajísima frecuencia
    - merequetengue = `vn(0,0,1,...)`

### Una solución: Steaming/Lematización (AKA la vieja confiable)

![](https://i.pinimg.com/736x/77/df/89/77df89e6ff57d332ba4e5d7bff723133--meme.jpg)

In [53]:
nltk.download("brown")

[nltk_data] Downloading package brown to /home/umoqnier/nltk_data...
[nltk_data]   Package brown is already up-to-date!


True

In [54]:
from nltk.corpus import brown

In [62]:
brown_corpus = preprocess(brown.words()[:100000], lang="en", remove_stops=True)

In [63]:
rprint(brown_corpus[:10])

['fulton', 'county', 'grand', 'jury', 'said', 'friday', 'investigation', 'atlantas', 'recent', 'primary']

In [ ]:
rprint(brown_corpus[:10])

['fulton', 'county', 'grand', 'jury', 'said', 'friday', 'investigation', 'atlantas', 'recent', 'primary']

In [66]:
from collections import Counter

rprint("[bright_yellow]Brown Vanilla")
rprint("Tokens:", len(brown.words()))
rprint("Tipos:", len(Counter(brown.words())))

rprint("[bright_green]Brown Preprocess")
rprint("Tokens:", len(brown_corpus))
rprint("Tipos:", len(Counter(brown_corpus)))

Brown Vanilla

Tokens: 1161192

Tipos: 56057

Brown Preprocess

Tokens: 50829

Tipos: 12601

#### Steamming

In [67]:
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("english")

In [68]:
stemmed_brown = [stemmer.stem(word) for word in brown_corpus]

#### Lematización

In [ ]:
!python -m spacy download en_core_web_md
!python -m spacy download es_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 20.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 12.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [71]:
import spacy


def lemmatize(words: list, lang: str = "en") -> list:
    model = "en_core_web_md" if lang == "en" else "es_core_news_md"
    nlp = spacy.load(model)
    nlp.max_length = 1500000
    lemmas = []
    for doc in nlp.pipe([" ".join(words)], batch_size=500):
        lemmas.extend([token.lemma_ for token in doc if not token.is_space])
    return lemmas

In [72]:
lemmatized_brown = lemmatize(brown_corpus, lang="en")

In [76]:
rprint("Tipos ([bright_magenta]word-based[/]):", len(Counter(brown_corpus)))
rprint("Tipos ([bright_yellow]Steamming[/]):", len(Counter(stemmed_brown)))
rprint("Tipos ([bright_green]Lemmatized[/]):", len(Counter(lemmatized_brown)))

Tipos (word-based): 12601

Tipos (Steamming): 8995

Tipos (Lemmatized): 10303

#### More problems?

<img src="https://uploads.dailydot.com/2019/10/Untitled_Goose_Game_Honk.jpeg?auto=compress%2Cformat&ixlib=php-3.3.0" width="250" height="250">

- Métodos dependientes de las lenguas
- Se pierde información
- Ruled-based

## Subword-tokenization salva el día 🦸🏼‍♀️

- Segmentación de palabras en unidades más pequeñas (*sub-words*)
- Obtenemos tipos menos variados y con mayores frecuencias
    - Esto le gusta modelos basados en métodos estadísticos
- Palabras frecuentes no deberían separarse
- Palabras largas y raras debería descomponerse en sub-palabras significativas
- Los métodos estadisticos que no requieren conocimiento a priori de las lenguas

In [ ]:
text = "Let's do tokenization!"
result = ["Let's", "do", "token", "ization", "!"]
print(f"Objetivo: {text} -> {result}")

Objetivo: Let's do tokenization! -> ["Let's", 'do', 'token', 'ization', '!']


### Algoritmos

Existen varios algoritmos para hacer *subword-tokenization* como los que se listan a continuación:

- Byte-Pair Encoding (BPE)
- WordPiece
- Unigram

#### BPE

- Segmenmentación iterativa, comienza segmentando en secuencias de caracteres
- Junta los pares más frecuentes (*merge operation*)
- Termina cuando se llega al número de *merge operations* especificado o número de vocabulario deseado (*hyperparams*, depende de la implementación)
- Introducido en el paper: [Neural Machine Translation of Rare Words with Subword Units, (Sennrich et al., 2015)](https://arxiv.org/abs/1508.07909)

In [ ]:
%%HTML
<iframe width="960" height="515" src="https://www.youtube.com/embed/HEikzVL-lZU"></iframe>

### Implementación de BPE

In [114]:
corpus = clean_and_extract_sentences(moby)

In [115]:
from nltk.tokenize import word_tokenize

In [116]:
from collections import defaultdict


word_freqs = defaultdict(int)

for sent in corpus:
    words = word_tokenize(sent)
    for word in words:
        word_freqs[word] += 1

In [117]:
print(word_freqs)

defaultdict(<class 'int'>, {'Loomings': 1, 'Call': 2, 'me': 621, 'Ishmael': 19, 'Some': 37, 'years': 89, 'ago': 35, '--': 1594, 'never': 192, 'mind': 78, 'how': 185, 'long': 303, 'precisely': 27, 'having': 61, 'little': 245, 'or': 676, 'no': 479, 'money': 11, 'in': 3830, 'my': 563, 'purse': 7, ',': 18913, 'and': 5931, 'nothing': 101, 'particular': 49, 'to': 4444, 'interest': 17, 'on': 987, 'shore': 18, 'I': 2108, 'thought': 147, 'would': 421, 'sail': 77, 'about': 302, 'a': 4465, 'see': 246, 'the': 13522, 'watery': 24, 'part': 148, 'of': 6402, 'world': 163, 'It': 302, 'is': 1675, 'way': 261, 'have': 749, 'driving': 9, 'off': 207, 'spleen': 1, 'regulating': 1, 'circulation': 1, 'Whenever': 1, 'find': 53, 'myself': 67, 'growing': 9, 'grim': 11, 'mouth': 54, ';': 4141, 'whenever': 14, 'it': 2188, 'damp': 8, 'drizzly': 1, 'November': 2, 'soul': 82, 'involuntarily': 12, 'pausing': 10, 'before': 290, 'coffin': 32, 'warehouses': 2, 'bringing': 10, 'up': 487, 'rear': 22, 'every': 215, 'funeral'

In [118]:
alphabet = set()

for word in word_freqs.keys():
    for char in word:
        if char not in alphabet:
            alphabet.add(char)

In [119]:
print(alphabet)

{'T', '2', 'b', "'", 'Y', 'F', 's', 'd', 'g', 'c', '7', 'v', '0', 'k', 'u', 'o', 'U', '1', '5', '6', 'W', 't', 'G', 'P', 'z', 'K', 'i', 'm', 'a', '`', ':', 'x', 'h', ';', ')', '&', 'X', 'p', 'j', '9', '4', 'q', 'n', 'J', 'f', 'B', 'w', 'D', 'E', 'R', 'r', 'N', 'M', '?', 'A', 'e', 'O', 'V', 'Q', '8', 'L', 'Z', ',', '3', 'C', 'H', '!', '*', 'l', '.', 'y', '(', 'I', '-', 'S'}


In [120]:
splits = {word: [c for c in word] for word in word_freqs.keys()}

In [121]:
print(splits)

{'Loomings': ['L', 'o', 'o', 'm', 'i', 'n', 'g', 's'], 'Call': ['C', 'a', 'l', 'l'], 'me': ['m', 'e'], 'Ishmael': ['I', 's', 'h', 'm', 'a', 'e', 'l'], 'Some': ['S', 'o', 'm', 'e'], 'years': ['y', 'e', 'a', 'r', 's'], 'ago': ['a', 'g', 'o'], '--': ['-', '-'], 'never': ['n', 'e', 'v', 'e', 'r'], 'mind': ['m', 'i', 'n', 'd'], 'how': ['h', 'o', 'w'], 'long': ['l', 'o', 'n', 'g'], 'precisely': ['p', 'r', 'e', 'c', 'i', 's', 'e', 'l', 'y'], 'having': ['h', 'a', 'v', 'i', 'n', 'g'], 'little': ['l', 'i', 't', 't', 'l', 'e'], 'or': ['o', 'r'], 'no': ['n', 'o'], 'money': ['m', 'o', 'n', 'e', 'y'], 'in': ['i', 'n'], 'my': ['m', 'y'], 'purse': ['p', 'u', 'r', 's', 'e'], ',': [','], 'and': ['a', 'n', 'd'], 'nothing': ['n', 'o', 't', 'h', 'i', 'n', 'g'], 'particular': ['p', 'a', 'r', 't', 'i', 'c', 'u', 'l', 'a', 'r'], 'to': ['t', 'o'], 'interest': ['i', 'n', 't', 'e', 'r', 'e', 's', 't'], 'on': ['o', 'n'], 'shore': ['s', 'h', 'o', 'r', 'e'], 'I': ['I'], 'thought': ['t', 'h', 'o', 'u', 'g', 'h', 't'

#### Creando el modelo

In [122]:
def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq
    return pair_freqs

In [123]:
pair_freqs = compute_pair_freqs(splits)

In [124]:
for i, key in enumerate(pair_freqs.keys()):
    print(f"{key}: {pair_freqs[key]}")
    if i >= 5:
        break

('L', 'o'): 170
('o', 'o'): 2727
('o', 'm'): 3906
('m', 'i'): 1601
('i', 'n'): 19043
('n', 'g'): 9771


In [125]:
best_pair = ""
max_freq = None

for pair, freq in pair_freqs.items():
    if max_freq is None or max_freq < freq:
        best_pair = pair
        max_freq = freq

print(best_pair, max_freq)

('t', 'h') 29424


Aplicamos el merge más común

In [126]:
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                split = split[:i] + [a + b] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits

In [127]:
vocab = []
merges = {}
vocab_size = 500

while len(vocab) < vocab_size:
    pair_freqs = compute_pair_freqs(splits)
    best_pair = ""
    max_freq = None
    for pair, freq in pair_freqs.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    splits = merge_pair(*best_pair, splits)
    merges[best_pair] = best_pair[0] + best_pair[1]
    vocab.append(best_pair[0] + best_pair[1])

In [128]:
def tokenize(text):
    words = word_tokenize(text)
    splits = [[c for c in word] for word in words]
    for pair, merge in merges.items():
        for idx, split in enumerate(splits):
            i = 0
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2 :]
                else:
                    i += 1
            splits[idx] = split

    return sum(splits, [])

In [130]:
for token in tokenize("This is necessary for you my heaviest friendly friend"):
    print(token)

T
his
is
ne
c
ess
ary
for
you
my
hea
vi
est
f
ri
end
ly
f
ri
end


In [ ]:
!pip install transformers

In [175]:
SENTENCE = "Let's do this tokenization to enable hypermodernization on my tokens tokenized 👁️👁️👁️!!!"

In [176]:
from transformers import GPT2Tokenizer

bpe_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
rprint(bpe_tokenizer.tokenize(SENTENCE))

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[
    'Let',
    "'s",
    'Ġdo',
    'Ġthis',
    'Ġtoken',
    'ization',
    'Ġto',
    'Ġenable',
    'Ġhyper',
    'modern',
    'ization',
    'Ġon',
    'Ġmy',
    'Ġtokens',
    'Ġtoken',
    'ized',
    'ĠðŁĳ',
    'ģ',
    'ï¸ı',
    'ðŁĳ',
    'ģ',
    'ï¸ı',
    'ðŁĳ',
    'ģ',
    'ï¸ı',
    '!!!'
]

In [ ]:
encoded_tokens = bpe_tokenizer(SENTENCE)
rprint(encoded_tokens["input_ids"])

[
    5756,
    338,
    466,
    428,
    11241,
    1634,
    284,
    7139,
    8718,
    23922,
    1634,
    319,
    616,
    16326,
    11241,
    1143,
    50169,
    223,
    37929,
    41840,
    223,
    37929,
    41840,
    223,
    37929,
    10185
]

In [ ]:
rprint(bpe_tokenizer.decode(encoded_tokens["input_ids"]))

Let's do this tokenization to enable hypermodernization on my tokens tokenized 👁️👁️👁️!!!

- En realidad GPT-2 usa *Byte-Level BPE*
    - Evitamos vocabularios de inicio grandes (Ej: unicode)
    - Usamos bytes como vocabulario base
    - Evitamos *Out Of Vocabulary, OOV* (aka `[UKW]`)

#### WordPiece

- Descrito en el paper: [Japanese and Korean voice search, (Schuster et al., 2012) ](https://static.googleusercontent.com/media/research.google.com/ja//pubs/archive/37842.pdf)
- Similar a BPE, inicia el vocabulario con todos los caracteres y aprende los merges
- En contraste con BPE, no elige con base en los pares más frecuentes si no los pares que maximicen la probabilidad de aparecer en los datos una vez que se agregan al vocabulario

$$score(a_i,b_j) = \frac{f(a_i,b_j)}{f(a_i)f(b_j)}$$

- Esto quiere decir que evalua la perdida de realizar un *merge* asegurandoce que vale la pena hacerlo

- Algoritmo usado en `BERT`

In [ ]:
%%HTML
<iframe width="960" height="500" src="https://www.youtube.com/embed/qpv6ms_t_1A"></iframe>

In [177]:
from transformers import BertTokenizer

SENTENCE = "🌽" + SENTENCE + "🔥"
wp_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
rprint(wp_tokenizer.tokenize(SENTENCE))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[
    '[UNK]',
    "'",
    's',
    'do',
    'this',
    'token',
    '##ization',
    'to',
    'enable',
    'hyper',
    '##mo',
    '##dern',
    '##ization',
    'on',
    'my',
    'token',
    '##s',
    'token',
    '##ized',
    '[UNK]',
    '!',
    '!',
    '!',
    '[UNK]'
]

<center><img src="https://us-tuna-sounds-images.voicemod.net/9cf541d2-dd7f-4c1c-ae37-8bc671c855fe-1665957161744.jpg"></center>

In [ ]:
rprint(wp_tokenizer(SENTENCE))

{
    'input_ids': [
        101,
        100,
        1005,
        1055,
        2079,
        2023,
        19204,
        3989,
        2000,
        9585,
        23760,
        5302,
        25888,
        3989,
        2006,
        2026,
        19204,
        2015,
        19204,
        3550,
        100,
        999,
        999,
        999,
        100,
        102
    ],
    'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
}

#### Unigram

- Algoritmo de subpword tokenization introducido en el paper: [Subword Regularization: Improving Neural Network Translation Models with Multiple Subword Candidates (Kudo, 2018)](https://arxiv.org/pdf/1804.10959.pdf)
- En contraste con BPE o WordPiece, este algoritmo inicia con un vocabulario muy grande y va reduciendolo hasta llegar tener un vocabulario deseado
- En cada iteración se calcula la perdida de quitar cierto elemento del vocabulario
    - Se quitará `p%` elementos que menos aumenten la perdida en esa iteración
- El algoritmo termina cuando se alcanza el tamaño deseado del vocabulario

Sin embargo, *Unigram* no se usa por si mismo en algun modelo de Hugging Face:
> "Unigram is not used directly for any of the models in the transformers, but it’s used in conjunction with SentencePiece." - Hugging face guy

#### SentencePiece


- No asume que las palabras estan divididas por espacios
- Trata la entrada de texto como un *stream* de datos crudos. Esto incluye al espacio como un caractér a usar
- Utiliza BPE o Unigram para construir el vocabulario

In [ ]:
# https://github.com/google/sentencepiece#installation
!pip install sentencepiece

In [173]:
from transformers import XLNetTokenizer

tokenizer = XLNetTokenizer.from_pretrained("xlnet-base-cased")
rprint(tokenizer.tokenize(SENTENCE))

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

NameError: name 'SENTENCE' is not defined

#### Objetivo de los subword tokenizers


- Buscamos que modelos de redes neuronales tenga datos mas frecuentes
- Esto ayuda a que en principio "aprendan" mejor
- Reducir el numero de tipos
- Reducir el numero de OOV

### Vamos a tokenizar 🌈
![](https://i.pinimg.com/736x/58/6b/88/586b8825f010ce0e3f9c831f568aafa8.jpg)

In [143]:
BASE_PATH = "."
CORPORA_PATH = f"{BASE_PATH}/data/tokenization"
MODELS_PATH = f"{BASE_PATH}/models/sub-word"

#### Corpus en español: CESS

In [132]:
nltk.download("cess_esp")

[nltk_data] Downloading package cess_esp to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package cess_esp is already up-to-date!


True

In [133]:
from nltk.corpus import cess_esp

cess_words = cess_esp.words()

In [134]:
" ".join(cess_words[:30])

'El grupo estatal Electricité_de_France -Fpa- EDF -Fpt- anunció hoy , jueves , la compra del 51_por_ciento de la empresa mexicana Electricidad_Águila_de_Altamira -Fpa- EAA -Fpt- , creada por el japonés Mitsubishi_Corporation'

In [135]:
cess_plain_text = " ".join(preprocess(cess_words))

In [136]:
rprint(f"'{cess_plain_text[300:600]}'")

'que el proyecto para la construcción de altamira_2 al norte de tampico prevé la utilización de gas natural como 
combustible principal en una central de ciclo combinado que debe empezar funcionar en mayo_del_2002 la electricidad
producida pasará la red eléctrica pública de méxico en_virtud_de un acue'

In [138]:
cess_preprocessed_words = cess_plain_text.split()

In [144]:
with open(f"{CORPORA_PATH}/cess_plain.txt", "w") as f:
    f.write(cess_plain_text)

#### Corpus Inglés: Gutenberg

In [150]:
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [151]:
from nltk.corpus import gutenberg

gutenberg_words = gutenberg.words()[:200000]

In [152]:
rprint(" ".join(gutenberg_words[:30]))

[ Emma by Jane Austen 1816 ] VOLUME I CHAPTER I Emma Woodhouse , handsome , clever , and rich , with a comfortable 
home and happy disposition , seemed

In [153]:
gutenberg_plain_text = " ".join(preprocess(gutenberg_words))

rprint(gutenberg_plain_text[:100])

emma by jane austen 1816 volume chapter emma woodhouse handsome clever and rich with comfortable hom

In [154]:
gutenberg_preprocessed_words = gutenberg_plain_text.split()

In [155]:
with open(f"{CORPORA_PATH}/gutenberg_plain.txt", "w") as f:
    f.write(gutenberg_plain_text)

#### Tokenizando el español con Hugging face

In [159]:
from transformers import AutoTokenizer

spanish_tokenizer = AutoTokenizer.from_pretrained(
    "dccuchile/bert-base-spanish-wwm-uncased"
)

In [160]:
rprint(spanish_tokenizer.tokenize(cess_plain_text[1000:1400]))

[
    'tra',
    '##les',
    'eléctricas',
    'en',
    'méxico',
    'se',
    'quedaron',
    'con',
    'dos',
    'cada',
    'una',
    'río',
    '_',
    'bravo',
    'sal',
    '##tillo',
    'para',
    'la',
    'compañía',
    'francesa',
    'alta',
    '##mir',
    '##a',
    'tu',
    '##x',
    '##pá',
    '##n',
    'para',
    'la',
    'japonesa',
    'ed',
    '##f',
    'tiene',
    'previsto',
    'invertir',
    '19',
    '##4',
    'millones',
    'de',
    'euros',
    'fp',
    '##a',
    '1',
    '##8',
    '##6',
    'millones',
    'de',
    'dólares',
    'fp',
    '##t',
    'en',
    'la',
    'central',
    'de',
    'río',
    '_',
    'bravo',
    'con',
    'una',
    'potencia',
    'de',
    '4',
    '##95',
    'mega',
    '##va',
    '##tios',
    '13',
    '##4',
    'millones',
    'de',
    'euros',
    'fp',
    '##a',
    '2',
    '##8',
    'millones',
    'de',
    'dólares',
    'fp',
    '##t',
    'en',
    'sal',
    '##tillo',
    'que',
    'como',
    'la',
    'primera',
    'funcionará',
    'con',
    'gas',
    'natural',
    'cuya',
    'pot',
    '##e'
]

In [161]:
cess_types = Counter(cess_words)

In [162]:
rprint(cess_types.most_common(10))

[
    (',', 11420),
    ('de', 10234),
    ('la', 6412),
    ('.', 5866),
    ('que', 5552),
    ('el', 5199),
    ('en', 4340),
    ('y', 4235),
    ('*0*', 3883),
    ('"', 3038)
]

In [163]:
cess_tokenized = spanish_tokenizer.tokenize(cess_plain_text)
rprint(cess_tokenized[:10])
cess_tokenized_types = Counter(cess_tokenized)

Token indices sequence length is longer than the specified maximum sequence length for this model (205282 > 512). Running this sequence through the model will result in indexing errors


['el', 'grupo', 'estatal', 'electri', '##ci', '##té', '_', 'de', '_', 'france']

In [164]:
rprint(cess_tokenized_types.most_common(30))

[
    ('de', 11838),
    ('_', 10457),
    ('la', 7163),
    ('el', 6090),
    ('que', 5948),
    ('en', 5001),
    ('los', 3230),
    ('del', 2516),
    ('las', 1957),
    ('se', 1954),
    ('por', 1949),
    ('un', 1906),
    ('fp', 1651),
    ('con', 1639),
    ('una', 1474),
    ('no', 1428),
    ('para', 1360),
    ('su', 1336),
    ('##a', 1302),
    ('al', 1240),
    ('##s', 1210),
    ('es', 1012),
    ('##t', 932),
    ('como', 788),
    ('lo', 783),
    ('ha', 729),
    ('más', 724),
    ('##n', 611),
    ('a', 611),
    ('sus', 519)
]

In [166]:
cess_lemmatized_types = Counter(lemmatize(cess_words, lang="es"))

In [167]:
rprint(cess_lemmatized_types.most_common(30))

[
    ('el', 17900),
    (',', 11426),
    ('de', 10280),
    ('*', 7767),
    ('.', 5894),
    ('que', 5568),
    ('en', 4629),
    ('y', 4340),
    ('0', 3889),
    ('uno', 3632),
    ('él', 3572),
    ('"', 3043),
    ('a', 3006),
    ('ser', 2425),
    ('del', 2261),
    ('su', 1824),
    ('haber', 1725),
    ('con', 1545),
    ('por', 1526),
    ('no', 1354),
    ('para', 1313),
    ('-', 1240),
    ('al', 1012),
    ('este', 848),
    ('-Fpt-', 774),
    ('-Fpa-', 764),
    ('como', 724),
    ('más', 664),
    ('estar', 577),
    ('tener', 569)
]

In [168]:
rprint("CESS")
rprint(f"Tipos ([bright_magenta]word-base[/]): {len(cess_types)}")
rprint(f"Tipos ([bright_yellow]lemmatized[/]): {len(cess_lemmatized_types)}")
rprint(f"Tipos ([bright_green]sub-word[/]): {len(cess_tokenized_types)}")

CESS

Tipos (word-base): 25464

Tipos (lemmatized): 17898

Tipos (sub-word): 17852

#### Tokenizando para el inglés

In [169]:
gutenberg_types = Counter(gutenberg_words)

In [178]:
gutenberg_tokenized = wp_tokenizer.tokenize(gutenberg_plain_text)
gutenberg_tokenized_types = Counter(gutenberg_tokenized)

Token indices sequence length is longer than the specified maximum sequence length for this model (170782 > 512). Running this sequence through the model will result in indexing errors


In [180]:
rprint(gutenberg_tokenized_types.most_common(10))

[
    ('the', 5478),
    ('to', 5476),
    ('and', 5120),
    ('of', 4562),
    ('it', 2601),
    ('her', 2577),
    ('she', 2496),
    ('was', 2486),
    ('in', 2361),
    ('not', 2197)
]

In [181]:
gutenberg_lemmatized_types = Counter(lemmatize(gutenberg_preprocessed_words))

In [182]:
rprint(gutenberg_lemmatized_types.most_common(20))

[
    ('be', 8493),
    ('the', 5478),
    ('to', 5450),
    ('and', 5120),
    ('of', 4562),
    ('she', 3528),
    ('have', 3445),
    ('he', 2676),
    ('it', 2601),
    ('not', 2341),
    ('in', 2316),
    ('you', 2007),
    ('that', 1869),
    ('as', 1502),
    ('but', 1485),
    ('her', 1476),
    ('for', 1399),
    ('do', 1341),
    ('with', 1263),
    ('very', 1252)
]

In [183]:
rprint("Gutenberg")
rprint(f"Tipos ([bright_magenta]word-base[/]): {len(gutenberg_types)}")
rprint(f"Tipos ([bright_yellow]lemmatized[/]): {len(gutenberg_lemmatized_types)}")
rprint(f"Tipos ([bright_green]sub-word[/]): {len(gutenberg_tokenized_types)}")

Gutenberg

Tipos (word-base): 8112

Tipos (lemmatized): 5466

Tipos (sub-word): 6793

#### OOV: out of vocabulary

Palabras que se vieron en el entrenamiento pero no estan en el test

In [184]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    gutenberg_words, test_size=0.3, random_state=42
)
rprint(len(train_data), len(test_data))

140000 60000

In [185]:
s_1 = {"a", "b", "c", "d", "e"}
s_2 = {"a", "x", "y", "d"}
rprint(s_1 - s_2)
rprint(s_2 - s_1)

{'b', 'c', 'e'}

{'x', 'y'}

In [186]:
oov_test = set(test_data) - set(train_data)

In [187]:
for word in list(oov_test)[:3]:
    rprint(f"{word} in train: {word in set(train_data)}")

finishing in train: False

humouredly in train: False

horsewoman in train: False

In [ ]:
train_tokenized, test_tokenized = train_test_split(
    gutenberg_tokenized, test_size=0.3, random_state=42
)
rprint(len(train_tokenized), len(test_tokenized))

124016 53151

In [ ]:
oov_tokenized_test = set(test_tokenized) - set(train_tokenized)

In [ ]:
rprint("OOV ([yellow]word-base):", len(oov_test))
rprint("OOV ([green]sub-word):", len(oov_tokenized_test))

OOV (word-base): 1114

OOV (sub-word): 728

## Entrenando nuestro modelo con BPE
![](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmedia1.tenor.com%2Fimages%2Fd565618bb1217a7c435579d9172270d0%2Ftenor.gif%3Fitemid%3D3379322&f=1&nofb=1&ipt=9719714edb643995ce9d978c8bab77f5310204960093070e37e183d5372096d9&ipo=images)

In [ ]:
!pip install subword-nmt

In [147]:
!ls {CORPORA_PATH}

1988.24s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


cess_plain.txt


In [148]:
!cat {CORPORA_PATH}/gutenberg_plain.txt

1994.51s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


cat: ./data/tokenization/gutenberg_plain.txt: No such file or directory


In [188]:
!subword-nmt learn-bpe -s 300 < \
 {CORPORA_PATH}/gutenberg_plain.txt > \
  {MODELS_PATH}/gutenberg.model

2362.75s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


100%|#######################################| 300/300 [00:00<00:00, 1631.77it/s]


In [189]:
!echo "I need to process this sentence because tokenization can be useful" \
| subword-nmt apply-bpe -c {MODELS_PATH}/gutenberg.model

2373.33s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


I ne@@ ed to pro@@ c@@ ess this s@@ ent@@ ence be@@ ca@@ u@@ se to@@ k@@ en@@ i@@ z@@ ation c@@ an be u@@ se@@ fu@@ l


In [190]:
!subword-nmt learn-bpe -s 1500 < \
{CORPORA_PATH}/gutenberg_plain.txt > \
 {MODELS_PATH}/gutenberg_high.model

2386.29s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


100%|#####################################| 1500/1500 [00:00<00:00, 2543.98it/s]


In [191]:
!echo "I need to process this sentence because tokenization can be useful" \
| subword-nmt apply-bpe -c {MODELS_PATH}/gutenberg_high.model

2392.50s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


I need to pro@@ c@@ ess this s@@ ent@@ ence because to@@ k@@ en@@ i@@ z@@ ation can be u@@ se@@ ful


### Aplicandolo a otros corpus: La biblia 📖🇻🇦

In [192]:
BIBLE_FILE_NAMES = {
    "spa": "spa-x-bible-reinavaleracontemporanea",
    "eng": "eng-x-bible-kingjames",
}

In [193]:
import requests


def get_bible_corpus(lang: str) -> str:
    """Download bible file corpus from GitHub repo"""
    file_name = BIBLE_FILE_NAMES[lang]
    r = requests.get(
        f"https://raw.githubusercontent.com/ximenina/theturningpoint/main/Detailed/corpora/corpusPBC/{file_name}.txt.clean.txt"
    )
    return r.text


def write_plain_text_corpus(raw_text: str, file_name: str) -> None:
    """Write file text on disk"""
    with open(f"{file_name}.txt", "w") as f:
        f.write(raw_text)

#### Biblia en Inglés

In [194]:
eng_bible_plain_text = get_bible_corpus("eng")
eng_bible_words = eng_bible_plain_text.lower().replace("\n", " ").split()

In [195]:
print(eng_bible_words[:10])

['the', 'beginning', 'of', 'the', 'gospel', 'of', 'jesus', 'christ', ',', 'the']


In [196]:
len(eng_bible_words)

30963

In [197]:
eng_bible_types = Counter(eng_bible_words)

In [198]:
rprint(eng_bible_types.most_common(30))

[
    (',', 2684),
    ('and', 2134),
    ('the', 1456),
    ('.', 965),
    ('he', 715),
    ('of', 709),
    ('him', 617),
    ('to', 507),
    ('that', 489),
    ('unto', 445),
    ('they', 442),
    ('them', 376),
    (':', 369),
    ('in', 323),
    ('a', 308),
    ('said', 298),
    (';', 262),
    ('shall', 262),
    ('is', 260),
    ('it', 243),
    ('not', 240),
    ('be', 238),
    ('his', 237),
    ('for', 233),
    ('ye', 226),
    ('was', 208),
    ('with', 203),
    ('but', 202),
    ('?', 199),
    ('when', 198)
]

In [199]:
eng_bible_lemmas_types = Counter(lemmatize(eng_bible_words, lang="en"))

In [200]:
write_plain_text_corpus(eng_bible_plain_text, f"{CORPORA_PATH}/eng-bible")

In [201]:
!subword-nmt apply-bpe -c {MODELS_PATH}/gutenberg_high.model < \
 {CORPORA_PATH}/eng-bible.txt > \
 {CORPORA_PATH}/eng-bible-tokenized.txt

2424.04s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [202]:
with open(f"{CORPORA_PATH}/eng-bible-tokenized.txt", "r") as f:
    tokenized_data = f.read()
eng_bible_tokenized = tokenized_data.split()

In [203]:
rprint(eng_bible_tokenized[:10])

['T@@', 'he', 'beginning', 'of', 'the', 'go@@', 'sp@@', 'el', 'of', 'J@@']

In [204]:
len(eng_bible_tokenized)

46887

In [205]:
eng_bible_tokenized_types = Counter(eng_bible_tokenized)
len(eng_bible_tokenized_types)

1123

In [206]:
eng_bible_tokenized_types.most_common(30)

[(',', 2684),
 ('the', 1423),
 ('and', 1318),
 ('d', 1105),
 ('n@@', 977),
 ('.', 965),
 ('to', 955),
 ('A@@', 884),
 ('he', 725),
 ('of', 706),
 ('him', 617),
 ('un@@', 505),
 ('that', 478),
 ('they', 434),
 ('e@@', 414),
 ('them', 376),
 (':', 369),
 ('in', 350),
 ('o@@', 349),
 ('th', 337),
 ('a', 337),
 ('e', 304),
 ('said', 298),
 ('it', 294),
 ('t', 270),
 ('J@@', 266),
 ('a@@', 264),
 (';', 262),
 ('ed', 261),
 ('shall', 261)]

#### ¿Qué pasa si aplicamos el modelo aprendido con Gutenberg a otras lenguas?

In [207]:
spa_bible_plain_text = get_bible_corpus("spa")
spa_bible_words = spa_bible_plain_text.replace("\n", " ").lower().split()

In [208]:
spa_bible_words[:10]

['principio',
 'del',
 'evangelio',
 'de',
 'jesucristo',
 ',',
 'el',
 'hijo',
 'de',
 'dios']

In [209]:
len(spa_bible_words)

30073

In [210]:
spa_bible_types = Counter(spa_bible_words)
len(spa_bible_types)

3317

In [211]:
spa_bible_types.most_common(30)

[(',', 1946),
 ('y', 1169),
 ('.', 1099),
 ('de', 1009),
 ('que', 927),
 ('a', 858),
 ('los', 645),
 ('la', 599),
 ('el', 572),
 (':', 539),
 ('se', 489),
 ('en', 461),
 ('«', 423),
 ('»', 423),
 ('jesús', 422),
 ('lo', 367),
 ('no', 312),
 ('le', 293),
 ('les', 267),
 ('dijo', 252),
 ('con', 220),
 ('pero', 217),
 ('al', 214),
 ('¿', 196),
 ('?', 195),
 ('por', 194),
 ('para', 172),
 ('su', 171),
 ('del', 165),
 ('un', 159)]

In [212]:
spa_bible_lemmas_types = Counter(lemmatize(spa_bible_words, lang="es"))
len(spa_bible_lemmas_types)

2136

In [213]:
write_plain_text_corpus(spa_bible_plain_text, f"{CORPORA_PATH}/spa-bible")

In [214]:
!subword-nmt apply-bpe -c {MODELS_PATH}/gutenberg_high.model < \
 {CORPORA_PATH}/spa-bible.txt > \
 {CORPORA_PATH}/spa-bible-tokenized.txt

2450.54s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [215]:
with open(f"{CORPORA_PATH}/spa-bible-tokenized.txt", "r") as f:
    tokenized_text = f.read()
spa_bible_tokenized = tokenized_text.split()

In [216]:
spa_bible_tokenized[:10]

['P@@', 'r@@', 'in@@', 'ci@@', 'pi@@', 'o', 'de@@', 'l', 'ev@@', 'an@@']

In [217]:
len(spa_bible_tokenized)

71836

In [218]:
spa_bible_tokenized_types = Counter(spa_bible_tokenized)
len(spa_bible_tokenized_types)

509

In [219]:
spa_bible_tokenized_types.most_common(40)

[('a', 3780),
 ('s', 2653),
 ('o', 2390),
 (',', 1946),
 ('e', 1660),
 ('l@@', 1648),
 ('qu@@', 1269),
 ('es@@', 1193),
 ('y', 1143),
 ('.', 1099),
 ('de', 1095),
 ('d@@', 991),
 ('t@@', 959),
 ('i@@', 911),
 ('o@@', 881),
 ('er@@', 861),
 ('lo@@', 853),
 ('s@@', 736),
 ('an@@', 724),
 ('n', 718),
 ('u@@', 716),
 ('í@@', 703),
 ('do', 697),
 ('di@@', 697),
 ('m@@', 690),
 ('c@@', 666),
 ('e@@', 661),
 ('as', 660),
 ('r@@', 643),
 ('ó', 628),
 ('on', 625),
 ('en', 620),
 ('j@@', 595),
 ('se', 581),
 ('b@@', 580),
 ('an', 577),
 ('en@@', 577),
 ('ar@@', 574),
 ('es', 564),
 ('el', 551)]

### Type-token Ratio (TTR)

- Una forma de medir la variación del vocabulario en un corpus
- Este se calcula como $TTR = \frac{len(types)}{len(tokens)}$
- Puede ser útil para monitorear la variación lexica de un texto

In [223]:
rprint("Información de la biblia en Inglés")
rprint("Tokens:", len(eng_bible_words))
rprint("Types ([bright_magenta]word-base):", len(eng_bible_types))
rprint("Types ([bright_yellow]lemmatized)", len(eng_bible_lemmas_types))
rprint("Types ([bright_green]BPE):", len(eng_bible_tokenized_types))
rprint("TTR ([bright_magenta]word-base):", len(eng_bible_types) / len(eng_bible_words))
rprint("TTR ([bright_green]BPE):", len(eng_bible_tokenized_types) / len(eng_bible_tokenized))

Información de la biblia en Inglés

Tokens: 30963

Types (word-base): 2198

Types (lemmatized) 1763

Types (BPE): 1123

TTR (word-base): 0.07098795336369215

TTR (BPE): 0.023951201825665965

In [225]:
rprint("Bible Spanish Information")
rprint("Tokens:", len(spa_bible_words))
rprint("Types ([bright_magenta]word-base):", len(spa_bible_types))
rprint("Types ([bright_yellow]lemmatized)", len(spa_bible_lemmas_types))
rprint("Types ([bright_green]BPE):", len(spa_bible_tokenized_types))
rprint("TTR ([bright_magenta]word-base):", len(spa_bible_types) / len(spa_bible_words))
rprint("TTR ([bright_green]BPE):", len(spa_bible_tokenized_types) / len(spa_bible_tokenized))

Bible Spanish Information

Tokens: 30073

Types (word-base): 3317

Types (lemmatized) 2136

Types (BPE): 509

TTR (word-base): 0.11029827419944802

TTR (BPE): 0.007085583829834623